# 5. Serviços para disponibilização dos modelos de IA

**Requisito da vaga:** *Desenvolver serviços para disponibilização dos modelos de IA*.

Os modelos (similaridade e anomalia) são **expostos como serviço HTTP** — `POST /similar` e `POST /anomaly` — consumidos pela API Go e por este notebook. Stateless, pronto para escalar em contêineres (Azure Container Apps, Phase 16).

Implementação real: `python/ml_service/service.py`.

> **Pré-requisito:** banco local com dados (`make db && make migrate && make smoke`)
> e o ML service rodando (`make ml-run &`) — ou apenas `make demo`.
>
> Carregar o helper compartilhado (bootstrap de imports, leitores de dados,
> URLs dos serviços) da primeira célula. `notebooks/common.py`.


In [1]:
import sys
sys.path.insert(0, r'/home/carlinhoshk/dev/ETL-Telemetria-Transformadores')
sys.path.insert(0, r'/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/notebooks')
import common
import requests
import pandas as pd
pd.set_option('display.max_columns', None)
print('ML service em', common.ML_URL)

ML service em http://localhost:8081


## Serviço 1 — Similaridade (`POST /similar`)

Entrada: `target` + `candidates` (projetos). Saída: top-k com score.

In [2]:
r = requests.get(f'{common.API_URL}/transformers/TR-001', timeout=10)
target = r.json()
candidates = common.to_plain_records(common.pg_df('SELECT * FROM transformers'))
resp = requests.post(f'{common.ML_URL}/similar', json={
    'target': target, 'candidates': candidates, 'top_k': 5}, timeout=15)
resp.raise_for_status()
pd.DataFrame(resp.json()['results'])

,transformer_id,score
0,TR-001,1.0000
1,TR-018,0.5967
2,TR-039,0.5797
3,TR-033,0.5425
4,TR-037,0.4940


## Serviço 2 — Anomalia (`POST /anomaly`)

Entrada: telemetria de um transformador. Saída: predição de outlier por ponto (`anomaly: 1`) com score. Modelo: `IsolationForest`.

In [3]:
telemetry = common.pg_df('''
SELECT transformer_id, ts AS timestamp, load_percent, ambient_temperature_c,
       oil_temperature_c, winding_temperature_c, oil_level_percent
FROM measurements ORDER BY ts LIMIT 200
''')
payload = {'measurements': common.to_plain_records(telemetry)}
resp = requests.post(f'{common.ML_URL}/anomaly', json=payload, timeout=30)
resp.raise_for_status()
anomalies = pd.DataFrame(resp.json()['results'])
anomalies

,transformer_id,timestamp,anomaly,score
0,TR-001,2026-08-12T06:00:00+00:00,True,-0.7986
1,TR-001,2026-08-12T07:18:26+00:00,False,-0.4196
2,TR-002,2026-08-12T07:18:26+00:00,False,-0.4042
3,TR-003,2026-08-12T07:18:26+00:00,False,-0.3877
4,TR-001,2026-08-12T07:18:27+00:00,False,-0.4490
5,TR-002,2026-08-12T07:18:27+00:00,False,-0.4873
6,TR-003,2026-08-12T07:18:27+00:00,False,-0.4991
7,TR-001,2026-08-12T07:18:28+00:00,False,-0.4820
8,TR-002,2026-08-12T07:18:28+00:00,False,-0.3902
9,TR-003,2026-08-12T07:18:28+00:00,False,-0.4258


In [4]:
print('pontos analisados:', len(anomalies))
print('outliers detectados:', int(anomalies['anomaly'].sum()))
outliers = anomalies[anomalies['anomaly'] == 1]
outliers.head()

pontos analisados: 16
outliers detectados: 2


,transformer_id,timestamp,anomaly,score
0,TR-001,2026-08-12T06:00:00+00:00,True,-0.7986
14,TR-002,2026-08-12T07:18:30+00:00,True,-0.5374


## Robustez do serviço

- Erros de entrada → HTTP 400 com mensagem clara; rota desconhecida → 404.
- Stateless: cada chamada refita o scaler — simples de escalar/replicar.

In [5]:
r = requests.post(f'{common.ML_URL}/similar', json={'bad': 'payload'}, timeout=10)
print('status:', r.status_code)
print(r.json())

status: 400
{'error': "'target'"}


## Conclusão

- Modelos de IA disponibilizados como serviços HTTP versionados e consumíveis pela API Go — plataforma pronta para orquestração em contêineres.
- Caminho para Azure: rotas expostas + healthcheck (`GET /health`) = probes nativos de Container Apps (docs/azure.md).